# Introduction
This notebook demonstrates how to set up and run a quantized version of the Llama-3-8B model. We will begin with some basic setup and then proceed to load and use the model.

## 1. Initial Setup
First, let's print a simple message to ensure our environment is set up correctly.

In [7]:
print("Hello World")

Hello World


## 2. Checking System Memory
We will check the available system memory to ensure that we have enough resources to load and run the model. The following command outputs the total, free, and available memory in gigabytes.

In [2]:
# !pip show transformers
# !pip show torch
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'

MemTotal: 251.77 GB
MemFree: 162.49 GB
MemAvailable: 242.28 GB


## 3. Code Formatting and Linting
We use `black` for code formatting and `pylint` for linting to ensure our code is clean and follows best practices.


In [3]:
!black Llama-3-8B-quant.ipynb
!pylint Llama-3-8B-quant.ipynb

reformatted Llama-3-8B-quant.ipynb

All done! ✨ 🍰 ✨
1 file reformatted.
************* Module Llama-3-8B-quant
Llama-3-8B-quant.ipynb:30:0: C0301: Line too long (239/100) (line-too-long)
Llama-3-8B-quant.ipynb:119:0: C0301: Line too long (128/100) (line-too-long)
Llama-3-8B-quant.ipynb:120:0: C0301: Line too long (473/100) (line-too-long)
Llama-3-8B-quant.ipynb:171:0: C0301: Line too long (116/100) (line-too-long)
Llama-3-8B-quant.ipynb:175:0: C0301: Line too long (106/100) (line-too-long)
Llama-3-8B-quant.ipynb:199:0: C0301: Line too long (127/100) (line-too-long)
Llama-3-8B-quant.ipynb:1:0: C0114: Missing module docstring (missing-module-docstring)
Llama-3-8B-quant.ipynb:1:0: C0103: Module name "Llama-3-8B-quant" doesn't conform to snake_case naming style (invalid-name)
Llama-3-8B-quant.ipynb:1:0: W0104: Statement seems to have no effect (pointless-statement)
Llama-3-8B-quant.ipynb:23:22: E0602: Undefined variable 'null' (undefined-variable)
Llama-3-8B-quant.ipynb:35:22: E0602: Undefi

## 4. Loading Environment Variables
We load the Hugging Face token from an environment variable to authenticate our session. This token is necessary to access the model from the Hugging Face Hub.


In [4]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

# Load environment variables from the .env file
load_dotenv()

# Read the Hugging Face token from the environment variable
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

# Log in using the token
login(token=huggingface_token)

Hugging Face token loaded successfully.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /nfs/homedirs/daro/.cache/huggingface/token
Login successful


## 5. Checking CUDA Availability
We check if CUDA is available on the system. CUDA is essential for running the model on GPU, which significantly speeds up the computations.


In [ ]:
import torch

# Check CUDA availability
if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")

## 6. Setting Up the Pipeline
We set up a text generation pipeline using the Llama-3-8B model from Hugging Face. This pipeline will be used to generate text based on a given input.

### 6.1 Meta-Llama-3-8B

In [6]:
from transformers import pipeline
import torch

model_id = "meta-llama/Meta-Llama-3-8B"
# device = f"cuda:{0}"
device = "cuda"

pipe = pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device=device,
)

messages = [
    {"role": "system", "content": "You are a pirate chatbot who always responds in pirate speak!"},
    {"role": "user", "content": "Who are you?"},
]

terminators = [
    pipe.tokenizer.eos_token_id,
    pipe.tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = pipe(
    messages,
    max_new_tokens=256,
    eos_token_id=terminators,
    do_sample=True,
    temperature=0.6,
    top_p=0.9,
)
assistant_response = outputs[0]["generated_text"][-1]["content"]
print(assistant_response)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


OutOfMemoryError: CUDA out of memory. Tried to allocate 112.00 MiB. GPU 

### 6.2 TinyLlama-1.1B

In [9]:
import torch
from transformers import pipeline

pipe = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", torch_dtype=torch.bfloat16, device_map="auto")

# We use the tokenizer's chat template to format each message - see https://huggingface.co/docs/transformers/main/en/chat_templating
messages = [
    {
        "role": "system",
        "content": "You are a friendly chatbot who always responds in the style of a pirate",
    },
    {"role": "user", "content": "How many helicopters can a human eat in one sitting?"},
]
prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
outputs = pipe(prompt, max_new_tokens=256, do_sample=True, temperature=0.7, top_k=50, top_p=0.95)
print(outputs[0]["generated_text"])
# <|system|>
# You are a friendly chatbot who always responds in the style of a pirate.</s>
# <|user|>
# How many helicopters can a human eat in one sitting?</s>
# <|assistant|>
# ...

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

/nfs/students/daro/miniconda3/envs/env-quant-rel/lib/python3.12/site-packages/accelerate/utils/modeling.py:1393: UserWarning: Current model requires 1408 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

<|system|>
You are a friendly chatbot who always responds in the style of a pirate</s>
<|user|>
How many helicopters can a human eat in one sitting?</s>
<|assistant|>
I don't have information about the specific human population, but a human can eat anywhere from 10 to 15 servings of food in one sitting. However, eating too much can lead to stomach cramps, nausea, and diarrhea, so it's best to limit your intake to 10-15 servings per sitting.


## 6.3. AutoModelForCausalLM Generation for Llama-3-8B

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# More convenient for us, for quantization

model_name = "meta-llama/Meta-Llama-3-8B"
# model_name = "microsoft/Phi-3-vision-128k-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
input_text = "Once upon a time"
inputs = tokenizer(input_text, return_tensors="pt")
outputs = model.generate(**inputs)
generated_text = tokenizer.decode(outputs[0])
print("Generated text:", generated_text)

In [ ]:
results = {"input": input_text, "output": generated_text}
with open("results.json", "w") as f:
    json.dump(results, f)

## 7. Quantization

In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from accelerate import init_empty_weights
from accelerate.utils import BnbQuantizationConfig, load_and_quantize_model
from huggingface_hub import snapshot_download

# Load the Llama-3-8B model configuration
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
device = f"cuda:{0}"

# Initialize an empty model using init_empty_weights
with init_empty_weights():
    empty_model = AutoModelForCausalLM.from_pretrained(model_name, config_dict=None)

# Download model weights
weights_location = snapshot_download(repo_id=model_name)

# Set 8-bit quantization configuration
bnb_quantization_config = BnbQuantizationConfig(load_in_8bit=True, llm_int8_threshold=6)

# Quantize the model
quantized_model = load_and_quantize_model(empty_model, weights_location=weights_location, bnb_quantization_config=bnb_quantization_config, device_map="auto")

TypeError: PretrainedConfig.from_dict() got multiple values for argument 'config_dict'

In [ ]:
from transformers import AutoModelForCausalLM
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_8bit=(weight_quantization_bits == 8),
    load_in_4bit=(weight_quantization_bits == 4),
    llm_int8_threshold=6.0,
    llm_int8_skip_modules=["lm_head"],
    llm_int8_enable_fp32_cpu_offload=False,
    llm_int8_has_fp16_weight=False,
    # bnb_4bit_compute_dtype=torch.float32,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="fp4",
    bnb_4bit_use_double_quant=double_quant,
)

awq_config = AWQConfig()

smashed_model_bnb = AutoModelForCausalLM.from_pretrained(
    temp_dir, quantization_config=bnb_config, trust_remote_code=True
)

# Calibration Dataset needed - WikiText? Something else because data leakage? TODO: Explore
smashed_model_awq = AutoModelForCausalLM.from_pretrained(
    temp_dir, quantization_config=awq_config, trust_remote_code=True
)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Define model ID
model_id = "meta-llama/Meta-Llama-3-8B"

# Load tokenizer and model (assuming CUDA is available)
device = f"cuda:{0}"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

# Input text
input_text = "Once upon a time"

# Convert input text to tensor and move to device
inputs = tokenizer(input_text, return_tensors="pt").to(device)

# Generate text using beam search (modify parameters as needed)
generated_ids = model.generate(
    input_ids=inputs["input_ids"],
    max_length=512,  # Adjust maximum output length
    num_beams=5,  # Adjust number of beams for beam search
)

# Decode generated IDs back to text
generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

# Print generated text
print("Generated text:", generated_text)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

## 8. Loading WikiText

In [ ]:
import os
from pytorch_lightning import LightningDataModule
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer
from datasets import load_dataset


# TODO: This dataset merge all independent sentences and process them as a single batch sample.
#  This is not ideal since sometimes sentences might switch from a topic to another.
#  However, it was done similarly on Wanda and SparseGPT code.
class TextDataset(Dataset):
    def __init__(self, dataset, tokenizer, sequence_length=2048):
        self.tokenizer = tokenizer
        self.dataset=dataset
        self.texts = dataset["text"]
        tokenized_dataset = self.tokenizer(" ".join(dataset["text"]), return_tensors="pt")
        self.data = tokenized_dataset.input_ids[0, :-1]
        self.labels = tokenized_dataset.input_ids[0]
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.data) // self.sequence_length

    def __getitem__(self, index):
        start_index = index * self.sequence_length
        end_index = (index + 1) * self.sequence_length
        return self.data[start_index:end_index], self.labels[start_index + 1 : end_index + 1]


# TODO: This dataset look at each sentence individually as a batch sample.
#  This is not ideal since sometimes sentences might not switch from a topic to another.
# class TextDataset(Dataset):
#     def __init__(self, dataset, tokenizer, sequence_length=2048):
#         self.texts = dataset["text"]
#         self.tokenizer = tokenizer
#         tokenized_dataset = self.tokenizer(
#             self.texts, return_tensors="pt", truncation=True, padding=True, max_length=sequence_length
#         )
#         self.data = tokenized_dataset.input_ids
#         self.sequence_length = sequence_length
#
#     def __len__(self):
#         return len(self.data)
#
#     def __getitem__(self, index):
#         return self.data[index, :-1], self.data[index, 1:]


class WikiTextDataModule(LightningDataModule):
    def __init__(self, directory_dataset=os.getcwd(), batch_size=64, sequence_length=2048, tokenizer_name=None, seed=1):
        super().__init__()
        self.directory_dataset = directory_dataset
        self.batch_size = batch_size
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, legacy=False)
        self.sequence_length = sequence_length
        self.prepare_data()

    def prepare_data(self):
        # Load train, val, and test datasets
        self.train_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
        self.val_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")
        self.test_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")

    # Use for calibration data
    def train_dataloader(self, batch_size=None, sequence_length=None):
        if batch_size is None:
            batch_size = self.batch_size
        if sequence_length is None:
            sequence_length = self.sequence_length
        else:
            sequence_length = min(self.sequence_length, sequence_length)
        dataset = TextDataset(self.train_dataset, tokenizer=self.tokenizer, sequence_length=sequence_length)
        train_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
        return train_dataloader

    # At this moment we are not using it
    def val_dataloader(self, batch_size=None, sequence_length=None):
        if batch_size is None:
            batch_size = self.batch_size
        if sequence_length is None:
            sequence_length = self.sequence_length
        else:
            sequence_length = min(self.sequence_length, sequence_length)
        dataset = TextDataset(self.val_dataset, tokenizer=self.tokenizer, sequence_length=sequence_length)
        val_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
        return val_dataloader

    # Use for evaluating perplexity
    def test_dataloader(self, batch_size=None, sequence_length=None):
        if batch_size is None:
            batch_size = self.batch_size
        if sequence_length is None:
            sequence_length = self.sequence_length
        else:
            sequence_length = min(self.sequence_length, sequence_length)
        dataset = TextDataset(self.test_dataset, tokenizer=self.tokenizer, sequence_length=sequence_length)
        test_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
        return test_dataloader

## 9. Text Streamer

In [14]:
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)